In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from nbody_sim.api import Simulation, ureg

# 1. Initialize the simulation
sim = Simulation()

# 2. Create a massive central star
sim.add_bodies(
    positions=[[0.0, 0.0]] * ureg.astronomical_unit,
    velocities=[[0.0, 0.0]] * (ureg.astronomical_unit / ureg.nbody_time),
    masses=[1000.0] * ureg.solar_mass
)

# 3. Create a disc of 200 orbiting planets (similar to your Rust utils.rs)
np.random.seed(42)
num_planets = 200
angles = np.random.uniform(0, 2 * np.pi, num_planets)
radii = np.random.uniform(2.0, 10.0, num_planets)

# Convert polar to cartesian
pos_x = radii * np.cos(angles)
pos_y = radii * np.sin(angles)
positions = np.column_stack((pos_x, pos_y)) * ureg.astronomical_unit

# For circular orbits: v = sqrt(G * M / r). In our units, G=1!
speeds = np.sqrt(1000.0 / radii)
vel_x = -speeds * np.sin(angles)
vel_y = speeds * np.cos(angles)
velocities = np.column_stack((vel_x, vel_y)) * (ureg.astronomical_unit / ureg.nbody_time)

masses = np.ones(num_planets) * ureg.earth_mass

sim.add_bodies(positions, velocities, masses)

ModuleNotFoundError: No module named 'nbody_sim'

In [ ]:
# 1. Set up a dark-themed Matplotlib figure
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(8, 8), dpi=100)
ax.set_xlim(-15, 15)
ax.set_ylim(-15, 15)
ax.set_aspect('equal')
ax.axis('off') # Hide the grid and axes for a cleaner look

# 2. Create the scatter plot object
# The central star will be drawn larger, the planets smaller.
sizes = [100] + [5] * num_planets
scatter = ax.scatter(sim._positions[:, 0], sim._positions[:, 1], 
                     s=sizes, c='white', alpha=0.8, edgecolors='none')

# 3. The update function called every frame
def update(frame):
    # Step the simulation forward (using a small dt for smooth physics)
    sim.step(0.01 * ureg.nbody_time)
    
    # Update the scatter plot with the new raw numpy positions
    scatter.set_offsets(sim._positions)
    return scatter,

# 4. Compile the animation
# frames=200 means it will simulate 200 steps. interval=30 is ~30fps.
anim = FuncAnimation(fig, update, frames=200, interval=30, blit=True)

# 5. Render directly into the Jupyter output cell
plt.close() # Prevents a duplicate static plot from showing up
HTML(anim.to_jshtml())